# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from nba_api.stats.static import players

from src.config import *
from src.utils import *
from src.feature_builder import *
from src.feature_aggregation import *

#display full columns
pd.set_option('display.max_column', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_seq_items', None)
# pd.set_option('display.max_colwidth', 500)
# pd.set_option('expand_frame_repr', True)

In [2]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-11-07 23:44:21.661166


# 🔁 Chargement des fichiers

In [3]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str})


/tmp/ipykernel_1075656/1838672219.py:3: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})


In [4]:
games_file

'data/01_bronze/games/games_merged_all_seasons_2025-11-07_23-30-01.csv'

In [5]:
df_boxscores

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage
0,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1897,Metta,World Peace,M. World Peace,metta-world-peace,F,NaN,NaN,29:20,83.3,112.3,-28.9,-28.9,0.077,1.00,6.7,0.038,0.031,0.034,6.7,0.583,0.582,0.203,95.73,95.73,60.0,0.156,0.467,0.267,0.249,0.475,0.186,0.100,0.370,2,2,4,10,17,14,13,36,2,0,0.917,0.083,0.933,0.267,0.000,0.267,0.067,0.133,0.667,0.429,0.571,0.0,0.0,0.429,0.571,7,12,0,1,0.000,1,2,0.50,1,1,2,1,3,2,1,2,15,-14.0,0.350,0.267,0.0,0.200,0.125,0.167,0.125,0.059,0.080,0.125,0.071,0.75,0.667,0.400,0.143,0.000,0.300
1,0020000279,1610612741,Chicago,Bulls,CHI,bulls,2033,Marcus,Fizer,M. Fizer,marcus-fizer,F,NaN,NaN,37:09,90.3,101.4,-11.1,-11.1,0.087,0.50,11.8,0.111,0.128,0.120,23.5,0.300,0.368,0.181,92.38,92.38,72.0,0.043,0.475,0.217,0.223,0.477,0.297,0.156,0.361,2,4,2,6,19,14,10,42,3,0,1.000,0.000,0.750,0.000,0.000,0.250,0.250,0.250,0.750,0.333,0.667,0.0,0.0,0.333,0.667,3,10,0,0,0.000,2,2,1.00,4,5,9,2,0,0,4,2,8,-7.0,0.115,0.167,0.0,0.000,0.250,0.154,0.400,0.217,0.273,0.167,0.250,0.00,0.000,0.600,0.095,0.000,0.123
2,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1434,Dragan,Tarlac,D. Tarlac,dragan-tarlac,C,NaN,NaN,22:49,85.1,122.4,-37.3,-37.3,0.000,0.00,0.0,0.150,0.154,0.152,75.0,1.000,1.064,0.151,100.98,100.98,47.0,-0.007,0.486,0.286,0.259,0.500,0.260,0.095,0.364,0,4,0,2,16,10,19,34,0,0,1.000,0.000,0.500,0.000,0.000,0.000,0.500,0.000,0.500,0.000,1.000,0.0,0.0,0.000,1.000,1,1,0,0,0.000,2,2,1.00

In [6]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22010,1.610613e+09,BOS,Boston Celtics,0021000001,2010-10-26,BOS vs. MIA,W,239,88,32,69,0.464,8,16,0.500,16,25,0.640,8,34,42,25,6,4,18,19,8.0,2010-11
1,22010,1.610613e+09,PHX,Phoenix Suns,0021000002,2010-10-26,PHX @ POR,L,239,92,36,74,0.486,9,19,0.474,11,16,0.688,7,23,30,15,3,4,19,19,-14.0,2010-11
2,22010,1.610613e+09,HOU,Houston Rockets,0021000003,2010-10-26,HOU @ LAL,L,240,110,38,91,0.418,8,20,0.400,26,28,0.929,16,37,53,25,6,7,20,25,-2.0,2010-11
3,22010,1.610613e+09,POR,Portland Trail Blazers,0021000002,2010-10-26,POR vs. PHX,W,240,106,43,93,0.462,10,20,0.500,10,15,0.667,18,30,48,31,11,2,12,22,14.0,2010-11
4,22010,1.610613e+09,MIA,Miami Heat,0021000001,2010-10-26,MIA @ BOS,L,242,80,27,74,0.365,8,20,0.400,18,25,0.720,11,28,39,15,10,6,17,21,-8.0,2010-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64489,22025,1.610613e+09,NYK,New York Knicks,0022500175,2025-11-05,NYK vs. MIN,W,239,137,55,102,0.539,19,42,0.452,8,9,0.889,21,29,50,32,7,8,14,21,23.0,2025-26
64490,22025,1.610613e+09,DAL,Dallas Mavericks,0022500177,2025-11-05,DAL vs. NOP,L,240,99,37,89,0.416,10,32,0.313,15,20,0.750,10,34,44,18,11,7,15,15,-2.0,2025-26
64491,22025,1.610613e+09,SAC,Sacramento Kings,0022500181,2025-11-05,SAC vs. GSW,W,239,121,41,83,0.494,11,23,0.478,28,33,0.848,8,39,47,27,7,5,16,18,5.0,2025-26
64492,22025,1.610613e+09,LAC,LA Clippers,0022500182,2025-11-06,LAC @ PHX,L,241,102,35,79,0.443,10,31,0.323,22,30,0.733,11,30,41,24,8,4,15,19,-13.0,2025-26


# 🧼 Nettoyage des minutes jouées

In [7]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['minutes'].apply(convert_minutes)

# Rename gameId and teamId in boxscores

In [8]:
rename_columns = {
    'gameId': 'GAME_ID',
    'teamId': 'TEAM_ID',
}

df_boxscores.rename(columns=rename_columns, inplace=True)


In [9]:
# cast 'GAME_ID', 'TEAM_ID' to string
df_games['GAME_ID'] = df_games['GAME_ID'].astype(str)
df_games['TEAM_ID'] = df_games['TEAM_ID'].astype(str)

df_boxscores['GAME_ID'] = df_boxscores['GAME_ID'].astype(str)
df_boxscores['TEAM_ID'] = df_boxscores['TEAM_ID'].astype(str)

In [10]:
print(df_games.dtypes)


SEASON_ID              int64
TEAM_ID               object
TEAM_ABBREVIATION     object
TEAM_NAME             object
GAME_ID               object
GAME_DATE             object
MATCHUP               object
WL                    object
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
SEASON                object
dtype: object


In [11]:
print(df_boxscores.dtypes)

GAME_ID                                object
TEAM_ID                                object
teamCity                               object
teamName                               object
teamTricode                            object
                                       ...   
percentageBlocksAllowed_usage         float64
percentagePersonalFouls_usage         float64
percentagePersonalFoulsDrawn_usage    float64
percentagePoints_usage                float64
MINUTES_PLAYED                        float64
Length: 101, dtype: object


# Merge GAME_DATE dans les boxscores et remove les duplicates créés

In [12]:
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
df_boxscores = df_boxscores.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

df_boxscores = df_boxscores.drop_duplicates(subset=['GAME_ID', 'TEAM_ID', 'playerSlug'])

In [13]:
# display nat GAME_DATE in df_boxscores
# check for NaT values in GAME_DATE
nat_dates = df_boxscores[df_boxscores['GAME_DATE'].isna()]
nat_dates

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE
0,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1897,Metta,World Peace,M. World Peace,metta-world-peace,F,NaN,NaN,29:20,83.3,112.3,-28.9,-28.9,0.077,1.00,6.7,0.038,0.031,0.034,6.7,0.583,0.582,0.203,95.73,95.73,60.0,0.156,0.467,0.267,0.249,0.475,0.186,0.100,0.370,2,2,4,10,17,14,13,36,2,0,0.917,0.083,0.933,0.267,0.000,0.267,0.067,0.133,0.667,0.429,0.571,0.0,0.0,0.429,0.571,7,12,0,1,0.000,1,2,0.50,1,1,2,1,3,2,1,2,15,-14.0,0.350,0.267,0.0,0.200,0.125,0.167,0.125,0.059,0.080,0.125,0.071,0.75,0.667,0.400,0.143,0.000,0.300,29.333333,NaT
1,0020000279,1610612741,Chicago,Bulls,CHI,bulls,2033,Marcus,Fizer,M. Fizer,marcus-fizer,F,NaN,NaN,37:09,90.3,101.4,-11.1,-11.1,0.087,0.50,11.8,0.111,0.128,0.120,23.5,0.300,0.368,0.181,92.38,92.38,72.0,0.043,0.475,0.217,0.223,0.477,0.297,0.156,0.361,2,4,2,6,19,14,10,42,3,0,1.000,0.000,0.750,0.000,0.000,0.250,0.250,0.250,0.750,0.333,0.667,0.0,0.0,0.333,0.667,3,10,0,0,0.000,2,2,1.00,4,5,9,2,0,0,4,2,8,-7.0,0.115,0.167,0.0,0.000,0.250,0.154,0.400,0.217,0.273,0.167,0.250,0.00,0.000,0.600,0.095,0.000,0.123,37.150000,NaT
2,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1434,Dragan,Tarlac,D. Tarlac,dragan-tarlac,C,NaN,NaN,22:49,85.1,122.4,-37.3,-37.3,0.000,0.00,0.0,0.150,0.154,0.152,75.0,1.000,1.064,0.151,100.98,100.98,47.0,-0.007,0.486,0.286,0.259,0.500,0.260,0.095,0.364,0,4,0,2,16,10,19,34,0,0,1.000,0.000,0.500,0.000,0.000,0.000,0.500,0.000,0.500

In [14]:
missing = df_boxscores[df_boxscores['GAME_DATE'].isna()][['GAME_ID', 'TEAM_ID']].drop_duplicates()

print(f"Exemples de lignes avec GAME_DATE NaT:")
print(missing.head(10))

# On regarde s’ils existent dans df_games
merged_check = missing.merge(df_games[['GAME_ID', 'TEAM_ID']], on=['GAME_ID', 'TEAM_ID'], how='left', indicator=True)
print(merged_check['_merge'].value_counts())


Exemples de lignes avec GAME_DATE NaT:
        GAME_ID     TEAM_ID
0    0020000279  1610612741
12   0020000279  1610612742
24   0020000282  1610612745
36   0020000282  1610612758
48   0020000281  1610612761
60   0020000281  1610612744
72   0020000283  1610612755
84   0020000283  1610612757
96   0020000278  1610612746
108  0020000278  1610612766
_merge
left_only     63954
right_only        0
both              0
Name: count, dtype: int64


In [15]:
print("Exemples d’ID manquants dans df_games")
print(missing[~missing.set_index(['GAME_ID', 'TEAM_ID']).index.isin(df_games.set_index(['GAME_ID', 'TEAM_ID']).index)])


Exemples d’ID manquants dans df_games
            GAME_ID     TEAM_ID
0        0020000279  1610612741
12       0020000279  1610612742
24       0020000282  1610612745
36       0020000282  1610612758
48       0020000281  1610612761
...             ...         ...
1507294  0022500180  1610612757
1507309  0022500175  1610612750
1507324  0022500175  1610612752
1507338  0022500182  1610612746
1507351  0022500182  1610612756

[63954 rows x 2 columns]


# 🔄 Cast dynamique des colonnes numériques


In [16]:
numeric_cols = df_boxscores.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

In [17]:
df_boxscores

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE
0,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1897,Metta,World Peace,M. World Peace,metta-world-peace,F,NaN,0.0,29:20,83.3,112.3,-28.9,-28.9,0.077,1.00,6.7,0.038,0.031,0.034,6.7,0.583,0.582,0.203,95.73,95.73,60.0,0.156,0.467,0.267,0.249,0.475,0.186,0.100,0.370,2,2,4,10,17,14,13,36,2,0,0.917,0.083,0.933,0.267,0.000,0.267,0.067,0.133,0.667,0.429,0.571,0.0,0.0,0.429,0.571,7,12,0,1,0.000,1,2,0.50,1,1,2,1,3,2,1,2,15,-14.0,0.350,0.267,0.0,0.200,0.125,0.167,0.125,0.059,0.080,0.125,0.071,0.75,0.667,0.400,0.143,0.000,0.300,29.333333,NaT
1,0020000279,1610612741,Chicago,Bulls,CHI,bulls,2033,Marcus,Fizer,M. Fizer,marcus-fizer,F,NaN,0.0,37:09,90.3,101.4,-11.1,-11.1,0.087,0.50,11.8,0.111,0.128,0.120,23.5,0.300,0.368,0.181,92.38,92.38,72.0,0.043,0.475,0.217,0.223,0.477,0.297,0.156,0.361,2,4,2,6,19,14,10,42,3,0,1.000,0.000,0.750,0.000,0.000,0.250,0.250,0.250,0.750,0.333,0.667,0.0,0.0,0.333,0.667,3,10,0,0,0.000,2,2,1.00,4,5,9,2,0,0,4,2,8,-7.0,0.115,0.167,0.0,0.000,0.250,0.154,0.400,0.217,0.273,0.167,0.250,0.00,0.000,0.600,0.095,0.000,0.123,37.150000,NaT
2,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1434,Dragan,Tarlac,D. Tarlac,dragan-tarlac,C,NaN,0.0,22:49,85.1,122.4,-37.3,-37.3,0.000,0.00,0.0,0.150,0.154,0.152,75.0,1.000,1.064,0.151,100.98,100.98,47.0,-0.007,0.486,0.286,0.259,0.500,0.260,0.095,0.364,0,4,0,2,16,10,19,34,0,0,1.000,0.000,0.500,0.000,0.000,0.000,0.500,0.000,0.500

# Build player features (players and top players absence for each game)

In [18]:
player_features_df = build_player_status_features(df_boxscores)

today = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


os.makedirs(DATA_PLAYERS_DIR, exist_ok=True)
player_features_output = os.path.join(DATA_PLAYERS_DIR, f'player_features_{today}.csv')

# to csv
#player_features_df.to_csv(player_features_output, index=False)

In [19]:
player_features_df

,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
0,0020000279,1610612741,NaT,1897,1,0,0,0,0,0,0,22.9,5.5,18.5,0.156,
1,0020000279,1610612741,NaT,2033,1,0,0,0,0,0,0,17.8,-2.0,8.6,0.043,
2,0020000279,1610612741,NaT,1434,1,0,0,0,0,0,0,7.4,-3.5,-1.2,-0.007,
3,0020000279,1610612741,NaT,2064,1,0,0,0,0,0,0,28.8,-0.5,26.6,0.188,
4,0020000279,1610612741,NaT,1500,1,0,0,0,0,0,0,20.3,0.5,17.7,0.037,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1507359,0022500182,1610612756,NaT,1630692,1,0,0,0,0,0,0,0.2,-2.0,-1.0,-0.265,
1507360,0022500182,1610612756,NaT,1630208,1,0,0,0,0,0,0,4.6,0.0,1.6,0.063,
1507361,0022500182,1610612756,NaT,1642863,1,0,0,0,0,0,0,0.0,0.0,0.0,0.000,
1507362,0022500182,1610612756,NaT,1642853,1,0,0,0,0,0,0,1.2,-1.0,0.0,0.000,


In [20]:
#print duplicates on 'GAME_ID' and 'TEAM_ID', personId
duplicates = player_features_df[player_features_df.duplicated(subset=['GAME_ID', 'TEAM_ID', 'personId'], keep=False)]
if not duplicates.empty:
    print("Duplicates found:")
    display(duplicates)

In [21]:
#player_features_df[player_features_df['is_absent'] == 1].sort_values(by='GAME_ID').head(50)

#same but filter with comment "DNP - Coach's Decision"

filtered = player_features_df[
    (player_features_df['is_absent'] == 1) &
    (~player_features_df['comment'].str.contains("coach's decision", na=False))
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(10))

filtered = player_features_df[
    (player_features_df['is_personal'] == 1) &
    (~player_features_df['comment'].str.contains("personal", na=False)) &
    (~player_features_df['comment'].str.contains("not with team", na=False)) 
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(10))




# display(player_features_df[
#     player_features_df['is_personal'] == 1 &
#     (~player_features_df['comment'].str.contains("nwt", na=False))
#     ].head(50))


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
1366614,0052300201,1610612741,NaT,1641763,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1366615,0052300201,1610612741,NaT,1630172,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1366628,0052300201,1610612748,NaT,202710,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1366629,0052300201,1610612748,NaT,1626196,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1366630,0052300201,1610612748,NaT,1626179,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1435671,0052400101,1610612737,NaT,1630552,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnp - injury/illness
1435725,0052400111,1610612748,NaT,1631107,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1435727,0052400111,1610612748,NaT,201567,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd-return to competition reconditioning
1435792,0052400201,1610612737,NaT,1630552,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
1435780,0052400201,1610612748,NaT,201567,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - personal


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
645998,0021200144,1610612738,NaT,2545,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
647652,0021200607,1610612745,NaT,202962,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
673094,0021201169,1610612756,NaT,201563,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
657429,0021201182,1610612739,NaT,203079,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
801619,0021400441,1610612757,NaT,2549,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family
791352,0021400871,1610612737,NaT,201143,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
83720,0040100154,1610612759,NaT,1495,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - death in family
203566,0040400311,1610612759,NaT,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business
203580,0040400312,1610612759,NaT,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business
203617,0040400313,1610612759,NaT,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business


In [22]:

# Analyse des statuts des joueurs
total_rows = len(player_features_df)

# Comptage des cas
n_present = player_features_df['is_present'].sum()
n_absent = player_features_df['is_absent'].sum()
n_injured = player_features_df['is_injured'].sum()

# is_resting
# is_suspended
# is_personal

n_resting = player_features_df['is_resting'].sum()
n_suspended = player_features_df['is_suspended'].sum()
n_personal = player_features_df['is_personal'].sum()

# Lignes incohérentes : aucune des colonnes n'est True

mask_no_status = (
    (player_features_df['is_present'] == 0) &
    (player_features_df['is_absent'] == 0) &
    (player_features_df['is_injured'] == 0) &
    (player_features_df['is_resting'] == 0) &
    (player_features_df['is_suspended'] == 0) &
    (player_features_df['is_personal'] == 0)
)


n_inconsistent = mask_no_status.sum()

# Lignes avec plusieurs statuts à la fois (logiquement impossible)
mask_multiple_status = (
    player_features_df[['is_present', 'is_absent', 'is_injured']].sum(axis=1) > 1
)
n_multiple = mask_multiple_status.sum()

print(f"✅ Analyse des statuts des joueurs sur {total_rows} lignes")
print(f" - Joueurs présents : {n_present}")
print(f" - Joueurs absents : {n_absent}")
print(f" - Joueurs blessés : {n_injured}")
print(f" - Joueurs en repos : {n_resting}")
print(f" - Joueurs suspendus : {n_suspended}")
print(f" - Joueurs pour raisons personnelles : {n_personal}")
print(f"❌ Lignes sans statut défini : {n_inconsistent}")
print(f"⚠️ Lignes avec plusieurs statuts actifs : {n_multiple}")

display(player_features_df[mask_no_status].head(10))



✅ Analyse des statuts des joueurs sur 755300 lignes
 - Joueurs présents : 619955
 - Joueurs absents : 135345
 - Joueurs blessés : 21345
 - Joueurs en repos : 392
 - Joueurs suspendus : 1039
 - Joueurs pour raisons personnelles : 1005
❌ Lignes sans statut défini : 0
⚠️ Lignes avec plusieurs statuts actifs : 21345


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment


# ⚙️ Aggrégation des data Player par équipe et match


In [23]:
# df_team_players = aggregate_team_player_features(player_features_df, top_n=10)

# display(df_team_players.head(10))
# display(df_team_players.tail(10))

# #display some lines with absent players
# absent_players = df_team_players[df_team_players['num_injured'] >= 1]
# absent_players 

# Identification des top joueurs sur l'ensemble du dataset
df_top_players = identify_historical_top_players(player_features_df)
print(f"✅ {df_top_players['is_historical_top'].sum()} top joueurs identifiés sur {len(df_top_players)} joueurs.")


df_agg_actual = aggregate_actual_team_features(player_features_df)
print(f"✅ {df_agg_actual.shape[0]} lignes générées dans l'aggregation actuelle par équipe.")

df_agg_top_abs = flag_top_players_absences(player_features_df, df_top_players)
print(f"✅ {df_agg_top_abs['top_player_absent'].sum()} absences de top joueurs détectées.")

df_team_features_final = df_agg_actual.merge(df_agg_top_abs, on=["GAME_ID", "TEAM_ID"], how="left")
df_team_features_final.fillna(0, inplace=True)

#add flags has_top_absent flag to use it as input and not roll it to see in analyse how much it helps to scrap this data before match
df_team_features_final['has_top_absent'] = df_team_features_final['top_player_absent'].apply(lambda x: 1 if x > 0 else 0)
df_team_features_final['has_absent'] = df_team_features_final['num_absent'].apply(lambda x: 1 if x > 0 else 0)


#sort by GAME_ID, TEAM_ID and GAME_DATE
df_team_features_final.sort_values(by=['GAME_ID', 'TEAM_ID', 'GAME_DATE'], inplace=True)


print(f"✅ Fusion réussie. Shape finale : {df_team_features_final.shape}")


/home/ju/Documents/Dev/NBA_Predictor/src/feature_aggregation.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tops = roll.groupby(['GAME_ID', 'TEAM_ID']).apply(pick_top).reset_index()


✅ 308495 top joueurs identifiés sur 308495 joueurs.
✅ 0 lignes générées dans l'aggregation actuelle par équipe.
✅ 22048 absences de top joueurs détectées.
✅ Fusion réussie. Shape finale : (0, 27)


In [24]:
# display(df_team_features_final.head(10))
# display(df_team_features_final.tail(10))

# top_player_absence_rate
# top_player_injury_rate
# top_player_resting_rate
# top_player_suspension_rate
# top_player_personal_rate

# display(df_team_features_final[df_team_features_final['top_player_absent_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_injury_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_resting_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_suspension_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_personal_rate'] > 0])
display(df_team_features_final)


,GAME_ID,TEAM_ID,GAME_DATE,player_perf_score_mean,player_perf_score_sum,num_present,num_absent,num_injured,num_suspended,num_resting,num_personal,num_absent_other,top_player_count,top_player_absent,top_player_injured,top_player_resting,top_player_suspended,top_player_personal,top_player_absent_other,top_player_absent_rate,top_player_injury_rate,top_player_resting_rate,top_player_suspension_rate,top_player_personal_rate,top_player_absent_other_rate,has_top_absent,has_absent


# ⚙️ Aggrégation des data Boxscore purs par équipe et match


## 📊 Agrégation des données par équipe et match


In [25]:

# 1. Agrégation par somme
group_keys = ['GAME_ID', 'TEAM_ID']
sum_agg = df_boxscores[group_keys + cols_to_sum].copy()
sum_agg = sum_agg.groupby(group_keys).sum().reset_index()

# 2. Agrégation pondérée par les minutes jouées
weighted_agg = compute_weighted_mean_features(df_boxscores, group_keys, cols_to_weighted_avg, weight_col='MINUTES_PLAYED')

# 3. Fusion des deux agrégats
team_match_stats = pd.merge(sum_agg, weighted_agg, on=group_keys, how='left')

# 4. Identifier l'équipe adverse
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
def get_opponent(row):
    teams = teams_in_game.get(row['GAME_ID'], [])
    opps = [tid for tid in teams if tid != row['TEAM_ID']]
    return opps[0] if opps else None

team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(get_opponent, axis=1)


In [26]:
count = 0

for game_id, teams in teams_in_game.items():
    if len(teams) < 2:
        #print(f"Attention : GAME_ID {game_id} a moins de 2 équipes : {teams}")
        count += 1
        
print(f"Nombre de GAME_ID avec moins de 2 équipes : {count}")



Nombre de GAME_ID avec moins de 2 équipes : 392


# 🔁 Ajout des colonnes OPP_ avec les data de l'adversaire

In [27]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [28]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'fieldGoalsMade_traditional',
       'fieldGoalsAttempted_traditional', 'threePointersMade_traditional',
       'threePointersAttempted_traditional', 'freeThrowsMade_traditional',
       'freeThrowsAttempted_traditional', 'reboundsOffensive_traditional',
       'reboundsDefensive_traditional',
       ...
       'OPP_percentageReboundsTotal_usage', 'OPP_percentageAssists_usage',
       'OPP_percentageTurnovers_usage', 'OPP_percentageSteals_usage',
       'OPP_percentageBlocks_usage', 'OPP_percentageBlocksAllowed_usage',
       'OPP_percentagePersonalFouls_usage',
       'OPP_percentagePersonalFoulsDrawn_usage', 'OPP_percentagePoints_usage',
       'OPP_plusMinusPoints_traditional'],
      dtype='object', length=165)

# 🏠 Ajout IS_HOME et IS_WIN


In [29]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22010,1610612738.0,BOS,Boston Celtics,0021000001,2010-10-26,BOS vs. MIA,W,239,88,32,69,0.464,8,16,0.500,16,25,0.640,8,34,42,25,6,4,18,19,8.0,2010-11
1,22010,1610612756.0,PHX,Phoenix Suns,0021000002,2010-10-26,PHX @ POR,L,239,92,36,74,0.486,9,19,0.474,11,16,0.688,7,23,30,15,3,4,19,19,-14.0,2010-11
2,22010,1610612745.0,HOU,Houston Rockets,0021000003,2010-10-26,HOU @ LAL,L,240,110,38,91,0.418,8,20,0.400,26,28,0.929,16,37,53,25,6,7,20,25,-2.0,2010-11
3,22010,1610612757.0,POR,Portland Trail Blazers,0021000002,2010-10-26,POR vs. PHX,W,240,106,43,93,0.462,10,20,0.500,10,15,0.667,18,30,48,31,11,2,12,22,14.0,2010-11
4,22010,1610612748.0,MIA,Miami Heat,0021000001,2010-10-26,MIA @ BOS,L,242,80,27,74,0.365,8,20,0.400,18,25,0.720,11,28,39,15,10,6,17,21,-8.0,2010-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64489,22025,1610612752.0,NYK,New York Knicks,0022500175,2025-11-05,NYK vs. MIN,W,239,137,55,102,0.539,19,42,0.452,8,9,0.889,21,29,50,32,7,8,14,21,23.0,2025-26
64490,22025,1610612742.0,DAL,Dallas Mavericks,0022500177,2025-11-05,DAL vs. NOP,L,240,99,37,89,0.416,10,32,0.313,15,20,0.750,10,34,44,18,11,7,15,15,-2.0,2025-26
64491,22025,1610612758.0,SAC,Sacramento Kings,0022500181,2025-11-05,SAC vs. GSW,W,239,121,41,83,0.494,11,23,0.478,28,33,0.848,8,39,47,27,7,5,16,18,5.0,2025-26
64492,22025,1610612746.0,LAC,LA Clippers,0022500182,2025-11-06,LAC @ PHX,L,241,102,35,79,0.443,10,31,0.323,22,30,0.733,11,30,41,24,8,4,15,19,-13.0,2025-26


In [30]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [31]:
# Assurer le format datetime pour GAME_DATE
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])



# Merge avec les infos de match (MATCHUP, SEASON)
match_dataset = match_dataset.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON','GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

In [32]:

print("match_dataset after merge")
display(match_dataset)

match_dataset after merge


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [33]:

#display lines with nan values on matchup
print("Lines with NaN in MATCHUP:")
display(match_dataset[match_dataset['MATCHUP'].isna()])

Lines with NaN in MATCHUP:


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [34]:



# Définir si l'équipe joue à domicile
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs').astype(int)

# Calcul du résultat (win) et écart de points
match_dataset['IS_WIN'] = (match_dataset['points_traditional'] > match_dataset['OPP_points_traditional']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['points_traditional'] - match_dataset['OPP_points_traditional']

# Conversion GAME_DATE et tri
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)


ValueError: cannot convert float NaN to integer

In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

# SPECIAL FOR ODDS : Keep only season past 2010-11


In [ ]:
# # get first line with season 2010-11 and cut all lines before
# first_season = match_dataset[match_dataset['SEASON'] == '2010-11'].index[0]
# match_dataset = match_dataset.iloc[first_season:].reset_index(drop=True)
# match_dataset

# Merge Odds with dataset

In [ ]:
all_odds_df = merge_odds_csv_files(DATA_ODDS_HISTORY_DIR)

match_dataset = match_odds_with_dataset_test(all_odds_df, match_dataset)



------------------ Nombre de lignes sans cotes (home/away): 28120 ------------------


In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

# Merge match_dataset with df_team_features_final on TEAM and GAME

In [ ]:

#team id as str
match_dataset['TEAM_ID'] = match_dataset['TEAM_ID'].astype(str)
match_dataset['GAME_ID'] = match_dataset['GAME_ID'].astype(str)

#date as datetime
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])

df_team_features_final['TEAM_ID'] = df_team_features_final['TEAM_ID'].astype(str)
df_team_features_final['GAME_ID'] = df_team_features_final['GAME_ID'].astype(str)
df_team_features_final['GAME_DATE'] = pd.to_datetime(df_team_features_final['GAME_DATE'])



# Merge match_dataset with df_team_features_final on TEAM and GAME, keep GAME_DATE
match_dataset = match_dataset.merge(
    df_team_features_final[['GAME_ID', 'TEAM_ID', 'GAME_DATE'] + df_team_features_final.columns.difference(['GAME_ID', 'TEAM_ID', 'GAME_DATE']).tolist()],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)



# match_dataset_ = match_dataset.merge(
#     df_team_features_final,
#     on=['GAME_ID', 'TEAM_ID'],
#     how='left'
# )
#remove duplicates
match_dataset = match_dataset.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])



In [ ]:
#sort by GAME_DATE and GAME_ID
#match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)



#print GAME_DATE_x and GAME_DATE_y to check if they are the same
print("GAME_DATE_x and GAME_DATE_y are the same:", (match_dataset['GAME_DATE_x'] == match_dataset['GAME_DATE_y']).all())

#drop GAME_DATE_x and GAME_DATE_y to GAME_DATE
match_dataset['GAME_DATE'] = match_dataset['GAME_DATE_x']
match_dataset = match_dataset.drop(columns=['GAME_DATE_x', 'GAME_DATE_y'])



#display cols in common match_dataset and df_team_features_final
common_cols = set(match_dataset.columns).intersection(set(df_team_features_final.columns))
print("Common columns between match_dataset and df_team_features_final:")
display(common_cols)


GAME_DATE_x and GAME_DATE_y are the same: True
Common columns between match_dataset and df_team_features_final:


{'GAME_DATE',
 'GAME_ID',
 'TEAM_ID',
 'has_absent',
 'has_top_absent',
 'num_absent',
 'num_absent_other',
 'num_injured',
 'num_personal',
 'num_present',
 'num_resting',
 'num_suspended',
 'player_perf_score_mean',
 'player_perf_score_sum',
 'top_player_absent',
 'top_player_absent_other',
 'top_player_absent_other_rate',
 'top_player_absent_rate',
 'top_player_count',
 'top_player_injured',
 'top_player_injury_rate',
 'top_player_personal',
 'top_player_personal_rate',
 'top_player_resting',
 'top_player_resting_rate',
 'top_player_suspended',
 'top_player_suspension_rate'}

# Reorder columns for visualisation

In [ ]:
cols_first = ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN','POINT_DIFF','has_absent','has_top_absent','num_absent','top_player_absent', 'points_traditional','ODDS','OPP_ODDS']
other_cols = [col for col in match_dataset.columns if col not in cols_first]
match_dataset = match_dataset[cols_first + other_cols]
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# Add OPP cols for player stats

In [ ]:
opp_cols = [f"OPP_{col}" for col in cols_player_stats]
df_opp = match_dataset.rename(columns={col: f"OPP_{col}" for col in cols_player_stats})

match_dataset = pd.merge(
    match_dataset,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [ ]:
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# 🚀 Features avancées


In [ ]:

# Ajouter les features avancées aux stats de match
#match_dataset = add_advanced_boxscore_features(match_dataset)

# MOVED TO CONFIG
# features_to_roll = cols_to_sum + cols_to_weighted_avg
# features_to_roll += [f"OPP_{col}" for col in cols_to_sum + cols_to_weighted_avg]

# Calcul des features glissantes shiftées
match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], features_to_roll, N_LIST, method="ewm")

match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], top_player_features_to_roll, N_LIST_TOP, method="ewm", apply_log=True)


match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset, win_shifted_col="IS_WIN_SHIFTED")


# Calcul des jours de repos pour l'équipe et l'adversaire
match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")

# Avantage de repos
match_dataset["REST_ADVANTAGE"] = (
    match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]
)


match_dataset = compute_rolling_rest_advantage(
    match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST
)

match_dataset = compute_home_away_pts(
    match_dataset, "TEAM_ID", "IS_HOME", "points_traditional", "OPP_points_traditional", N_LIST
)

# match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
# match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)

match_dataset = convert_elos_to_elo_diff(match_dataset)


/home/ju/Documents/Dev/Python/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
/home/ju/Documents/Dev/Python/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
/home/ju/Documents/Dev/Python/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [ ]:
pd.options.display.max_columns = None
display(match_dataset)

GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0029900001 1999-11-02  1610612739  1610612752        0       0   
1      0029900001 1999-11-02  1610612752  1610612739        1       1   
2      0029900002 1999-11-02  1610612751  1610612754        1       0   
3      0029900002 1999-11-02  1610612754  1610612751        0       1   
4      0029900003 1999-11-02  1610612737  1610612764        0       0   
...           ...        ...         ...         ...      ...     ...   
66196  0042400401 2025-06-05  1610612760  1610612754        1       0   
66197  0042400402 2025-06-08  1610612754  1610612760        0       0   
66198  0042400402 2025-06-08  1610612760  1610612754        1       1   
66199  0042400403 2025-06-11  1610612754  1610612760        1       1   
66200  0042400403 2025-06-11  1610612760  1610612754        0       0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0            -8.0           1               0           1                  0   
1             8.0           1               0           3                  0   
2            -7.0           1               1           2                  1   
3             7.0           1               0           2                  0   
4            -7.0           1               1           2                  1   
...           ...         ...             ...         ...                ...   
66196        -1.0           1               0           3                  0   
66197       -16.0           1               0           3                  0   
66198        16.0           0               0           0                  0   
66199         9.0           1               0           5                  0   
66200        -9.0           1               0           3                  0   

       points_traditional  ODDS  OPP_ODDS  fieldGoalsMade_traditional  \
0                      84   NaN       NaN                          30   
1                      92   NaN       NaN                          32   
2                     112   NaN       NaN                          36   
3                     119   NaN       NaN                          37   
4                      87   NaN       NaN                          31   
...                   ...   ...       ...                         ...   
66196                 110   NaN       NaN                          39   
66197                 107   NaN       NaN                          37   
66198                 123   NaN       NaN                          40   
66199                 116   NaN       NaN                          44   
66200                 107   NaN       NaN                          37   

       fieldGoalsAttempted_traditional  threePointersMade_traditional  \
0                                   77                              7   
1                                   74                              6   
2                                   81                              5   
3                                   78                              6   
4                                   78                              2   
...                                ...                            ...   
66196                               98                             11   
66197                               82                             14   
66198                               82                             14   
66199                               85                              9   
66200                               79                             10   

       threePointersAttempted_traditional  freeThrowsMade_traditional  \
0                                      14                          17   
1                                      15                          22   
2                                      12                          35   
3                                      13                          39   
4                                       6            

# 🧽 Nettoyage et sauvegarde


In [ ]:

#TODO : see why not found in config.py after reboot
player_absent_input_cols = [
    'has_top_absent',
    'has_absent',
    'top_player_absent',
    'num_absent',
]

final_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

os.makedirs(DATA_FINAL_DATASET_DIR, exist_ok=True)
os.makedirs(DATA_FINAL_CLEANED_DATASET_DIR, exist_ok=True)

match_dataset.to_csv(raw_save_path, index=False)

cols_to_drop = features_to_roll+top_player_features_to_roll+COLS_MATCH_REAL
#remove num_absent and top_player_absent from cols_to_drop to see if we can use them has input
cols_to_drop = [col for col in cols_to_drop if col not in player_absent_input_cols]


final_cleaned = match_dataset.drop(columns=cols_to_drop, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

final_cleaned

✅ Fichier brut : data/final_dataset/nba_features_final_2025-06-18_18-06-14.csv
✅ Fichier clean : data/final_cleaned_dataset/nba_features_cleaned_final_2025-06-18_18-06-14.csv


GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0029900001 1999-11-02  1610612739  1610612752        0       0   
1      0029900001 1999-11-02  1610612752  1610612739        1       1   
2      0029900002 1999-11-02  1610612751  1610612754        1       0   
3      0029900002 1999-11-02  1610612754  1610612751        0       1   
4      0029900003 1999-11-02  1610612737  1610612764        0       0   
...           ...        ...         ...         ...      ...     ...   
66196  0042400401 2025-06-05  1610612760  1610612754        1       0   
66197  0042400402 2025-06-08  1610612754  1610612760        0       0   
66198  0042400402 2025-06-08  1610612760  1610612754        1       1   
66199  0042400403 2025-06-11  1610612754  1610612760        1       1   
66200  0042400403 2025-06-11  1610612760  1610612754        0       0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0            -8.0           1               0           1                  0   
1             8.0           1               0           3                  0   
2            -7.0           1               1           2                  1   
3             7.0           1               0           2                  0   
4            -7.0           1               1           2                  1   
...           ...         ...             ...         ...                ...   
66196        -1.0           1               0           3                  0   
66197       -16.0           1               0           3                  0   
66198        16.0           0               0           0                  0   
66199         9.0           1               0           5                  0   
66200        -9.0           1               0           3                  0   

       ODDS  OPP_ODDS   SEASON  ROLL_fieldGoalsMade_traditional_3  \
0       NaN       NaN  1999-00                                NaN   
1       NaN       NaN  1999-00                                NaN   
2       NaN       NaN  1999-00                                NaN   
3       NaN       NaN  1999-00                                NaN   
4       NaN       NaN  1999-00                                NaN   
...     ...       ...      ...                                ...   
66196   NaN       NaN  2024-25                          44.843719   
66197   NaN       NaN  2024-25                          40.028400   
66198   NaN       NaN  2024-25                          41.921860   
66199   NaN       NaN  2024-25                          38.514200   
66200   NaN       NaN  2024-25                          40.960930   

       ROLL_fieldGoalsMade_traditional_5  ROLL_fieldGoalsMade_traditional_10  \
0                                    NaN                                 NaN   
1                                    NaN                                 NaN   
2                                    NaN                                 NaN   
3                                    NaN                                 NaN   
4                                    NaN                                 NaN   
...                                  ...                                 ...   
66196                          43.983628                           43.278729   
66197                          40.234179                           41.051584   
66198                          42.322419                           42.500779   
66199                          39.156119                           40.314932   
66200                          41.548279                           42.046092   

       ROLL_fieldGoalsMade_traditional_25  ROLL_fieldGoalsMade_traditional_50  \
0                                     NaN                                 NaN   
1                                     NaN                                 NaN   
2                                     NaN                                 NaN   
3                                     NaN                 

In [ ]:
final_cleaned.columns

Index(['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN',
       'POINT_DIFF', 'has_absent', 'has_top_absent', 'num_absent',
       ...
       'H2H_LAST_100_COUNT', 'H2H_LAST_200_DIFF', 'H2H_LAST_200_WINRATE',
       'H2H_LAST_200_COUNT', 'H2H_SEASON_WINS', 'H2H_SEASON_MATCHES',
       'H2H_SEASON_WINRATE', 'H2H_WIN_STREAK', 'ELO_DIFF', 'ELO_DIFF_SEASON'],
      dtype='object', length=1464)

In [ ]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-18 18:14:18.253294
Total time:  0:20:51.035422
